# KIRA — 1. Préparer les données (CPU, sans GPU)

Ce notebook télécharge des textes français **aux révisions épinglées**, vérifie leur taille et leur empreinte
SHA-256, apprend le vocabulaire de KIRA sur le split d'entraînement seulement, puis écrit les fichiers de jetons.
Il ne consomme **pas** le quota GPU.

**Réglages Kaggle** (panneau de droite) : *Accelerator* = **None** ; *Internet* = **On** ;
*Environment* = **Pin to original environment**.
Puis **Save Version → Save & Run All (Commit)**. Tu peux fermer la page : le travail continue.

Sortie : le dossier `donnees/` (jetons, tokenizer, `manifest.json` avec licences et attributions).
Les sources et leurs licences sont décrites dans `docs/kira-kaggle.md` du dépôt.

In [ ]:
# Paramètres
KIRA_REF = "main"          # branche ou commit (40 caractères) du dépôt public
VOCAB_SIZE = 32000         # vocabulaire texte appris (les rôles s'ajoutent hors du texte)
SOURCES = ""               # vide = toutes les sources du registre ; sinon "nom1,nom2"
BUDGET = ""                # facultatif : "source=jetons,source=jetons" pour plafonner l'entraînement
TIME_LIMIT_HOURS = 10.5    # arrêt propre avant la limite de 12 h de Kaggle

In [ ]:
# Retrouve les entrées ajoutées au notebook (« Add Input »), en lecture seule sous /kaggle/input.
import json, os, shutil, subprocess, sys
from pathlib import Path

def find_data():
    found = []
    for manifest in Path("/kaggle/input").rglob("manifest.json"):
        try:
            if json.loads(manifest.read_text(encoding="utf-8")).get("format") == "kira-token-shards-1":
                found.append(manifest.parent)
        except (OSError, ValueError):
            pass
    if len(found) != 1:
        raise SystemExit("Il faut exactement une entrée de données KIRA (sortie du notebook kira-1-donnees). "
                         f"Trouvé : {[str(f) for f in found]}")
    return found[0]

def find_run(name):
    """Session la plus avancée d'un run parmi les entrées : (dossier, étape), ou (None, None)."""
    best, best_step = None, None
    for state in Path("/kaggle/input").rglob(f"runs/{name}/training.pt"):
        manifest = state.parent / "manifest.json"
        step = json.loads(manifest.read_text(encoding="utf-8")).get("steps", 0) if manifest.exists() else 0
        if best_step is None or step > best_step:
            best, best_step = state.parent, step
    return best, best_step

In [ ]:
# Récupère le code public de KIRA au commit choisi et installe le tokenizer épinglé.
def run(*command, **options):
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run([str(c) for c in command], check=True, **options)

CODE = Path("/tmp/KIRA")
if not (CODE / ".git").exists():
    run("git", "clone", "--quiet", "https://github.com/BlackGhost99/KIRA", CODE)
run("git", "-C", CODE, "fetch", "--quiet", "origin")
run("git", "-C", CODE, "checkout", "--quiet", "--detach", KIRA_REF if len(KIRA_REF) == 40 else "origin/" + KIRA_REF)
COMMIT = subprocess.run(["git", "-C", str(CODE), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
os.environ.update(KIRA_CODE_COMMIT=COMMIT, PYTHONPATH=str(CODE), PYTHONUNBUFFERED="1")
run(sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "tokenizers==0.23.3")
import torch
image = Path("/etc/git_commit")
print("Code KIRA :", COMMIT, "| PyTorch", torch.__version__, "| GPU :", torch.cuda.device_count(),
      "| image Kaggle :", image.read_text().strip() if image.exists() else "?")

In [ ]:
# Lance une commande et garde aussi sa sortie dans un fichier journal.
def stream(command, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("$", " ".join(map(str, command)), flush=True)
    with open(log_path, "a", encoding="utf-8") as log:
        process = subprocess.Popen([str(c) for c in command], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
    if code:
        raise SystemExit(f"Échec (code {code}) : voir {log_path}. Les sauvegardes déjà écrites restent dans /kaggle/working.")

In [ ]:
# Les fichiers bruts vont dans /kaggle/working (Kaggle peut arrêter une session qui écrit trop ailleurs)
# et sont effacés dès qu'ils sont convertis ; seul donnees/ reste dans la sortie.
OUTPUT, RAW = Path("/kaggle/working/donnees"), Path("/kaggle/working/brut")
command = [sys.executable, "-m", "research.kira_model.corpus", "prepare", "--output", OUTPUT, "--raw-dir", RAW,
           "--vocab-size", VOCAB_SIZE, "--threads", os.cpu_count() or 4,
           "--time-limit", int(TIME_LIMIT_HOURS * 3600), "--delete-raw"]
if SOURCES:
    command += ["--sources", SOURCES]
if BUDGET:
    command += ["--budget", BUDGET]
try:
    stream(command, Path("/kaggle/working/journal-donnees.txt"))
finally:
    shutil.rmtree(RAW, ignore_errors=True)

In [ ]:
# Résumé : jetons par source et par split, licences.
manifest = json.loads((OUTPUT / "manifest.json").read_text(encoding="utf-8"))
total = 0
for name, source in manifest["sources"].items():
    tokens = {split: part["tokens"] for split, part in source["splits"].items()}
    total += tokens["train"]
    print(f"{name:26} train {tokens['train']/1e9:6.2f} G  validation {tokens['validation']/1e6:5.2f} M  "
          f"test {tokens['test']/1e6:5.2f} M  | {source.get('license')}")
print(f"Total entraînement : {total/1e9:.2f} milliard(s) de jetons ; arrêt par limite de temps : {manifest['time_limit_reached']}")
run("du", "-sh", OUTPUT)